In [1]:
import os, sys, random, time, warnings, tarfile
import numpy as np
import torch
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from pathlib import Path
from collections import Counter
import scipy.io
import timm
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.environ["PYTORCH_ALLOC_CONF"]         = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(42)
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PLOT_DIR = '/kaggle/working/plots'
os.makedirs(PLOT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

RESULTS = {
    'exp1':        {'Aircraft':0.,'CUB':0.,'Flowers':0.,'Cars':0.,'Dogs':0.},
    'exp2_cub':    {100:{'Many':0.,'Medium':0.,'Few':0.,'All':0.},
                    50: {'All':0.}, 10: {'All':0.}},
    'exp2_flower': {100:{'Many':0.,'Medium':0.,'Few':0.,'All':0.},
                    50: {'All':0.}, 10: {'All':0.}},
    'exp3':        {1:0., 5:0., 10:0., 'all':0.},
    'exp4':        {'WB+Water':0.,'WB+Land':0.,'LB+Land':0.,'LB+Water':0.},
}
print("Results store ✓")

Device: cuda
Results store ✓


In [ ]:
MN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
MS = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

def denorm(x): return (x*MS.to(x.device,x.dtype)+MN.to(x.device,x.dtype)).clamp(0,1)
def renorm(x): return (x.clamp(0,1)-MN.to(x.device,x.dtype))/MS.to(x.device,x.dtype)

def split_params(model):
    hk=('fc','head','classifier'); bb,hd=[],[]
    for n,p in model.named_parameters():
        (hd if any(k in n for k in hk) else bb).append(p)
    return bb,hd

@torch.no_grad()
def soft_saliency(x_norm, keep_ratio):
    x    = denorm(x_norm)
    gray = 0.2989*x[:,0:1]+0.5870*x[:,1:2]+0.1140*x[:,2:3]
    kx   = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]],
                         device=x.device,dtype=x.dtype).view(1,1,3,3)
    lap  = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]],
                         device=x.device,dtype=x.dtype).view(1,1,3,3)
    sal  = (0.55*torch.sqrt(F.conv2d(gray,kx,padding=1)**2+
                            F.conv2d(gray,kx.transpose(-1,-2),padding=1)**2+1e-6)
            +0.45*F.conv2d(gray,lap,padding=1).abs())
    B,_,H,W = sal.shape
    yy,xx = torch.meshgrid(torch.linspace(-1,1,H,device=x.device),
                            torch.linspace(-1,1,W,device=x.device),indexing='ij')
    sal = sal*(0.55+0.45*torch.exp(-(xx**2+yy**2)/0.70
               ).view(1,1,H,W).to(x.dtype))
    sal = sal/(sal.flatten(1).amax(1).view(B,1,1,1)+1e-6)
    q   = torch.quantile(sal.flatten(1),1.0-keep_ratio,dim=1).view(B,1,1,1)
    alpha = torch.sigmoid(12.0*(sal-q))
    alpha = F.max_pool2d(alpha,5,stride=1,padding=2)
    return F.avg_pool2d(alpha,3,stride=1,padding=1).clamp(0,1)

@torch.no_grad()
def hdmix(imgs, labels, model, epoch, total_epochs, cfg):
    """
    Label-preserving. cfg passed explicitly — no global default.
    Skips automatically when epoch <= warmup_epochs.
    """
    if imgs.size(0)<2 or epoch<=cfg['warmup_epochs']:
        return imgs, labels

    alpha = soft_saliency(imgs, cfg['keep_ratio'])

    # Background-complementarity partner selection
    small  = F.interpolate(alpha,(28,28),mode='bilinear',align_corners=False).flatten(1)
    bg     = 1.0-small
    scores = torch.mm(bg,bg.t()); scores.fill_diagonal_(-1e9)
    K      = min(cfg['K'], imgs.size(0)-1)
    topk   = scores.topk(K,dim=1).indices
    idx    = topk[:,epoch%K]

    src = denorm(imgs); tgt = denorm(imgs[idx])

    # Entropy-adaptive strength
    flat   = alpha.flatten(1).clamp(1e-6,1-1e-6)
    H_n    = (-(flat*flat.log()+(1-flat)*(1-flat).log()).mean(1)/0.6931).clamp(0,1)
    prog   = min(1.0,(epoch-cfg['warmup_epochs'])/max(1,total_epochs-cfg['warmup_epochs']))
    s_ceil = cfg['s_min']+(cfg['s_max']-cfg['s_min'])*prog
    s      = (s_ceil-0.15*H_n).clamp(cfg['s_min'],s_ceil).view(-1,1,1,1)

    tgt_sm = F.avg_pool2d(tgt,5,stride=1,padding=2)
    bg_out = ((1-s)*tgt+s*tgt_sm
              +torch.randn_like(tgt)*(cfg['noise_scale']*s)).clamp(0,1)
    mixed  = renorm(alpha*src+(1-alpha)*bg_out)

    # Confidence filter (no extra forward pass overhead issue — uses model state)
    if epoch>=cfg['filter_epoch']:
        model.eval()
        with torch.no_grad(), autocast():
            prob = torch.softmax(model(mixed),dim=1)
        conf,pred = prob.max(1)
        keep = ((pred==labels)&(conf>=cfg['min_conf'])).float().view(-1,1,1,1)
        mixed = keep*mixed+(1-keep)*imgs
        model.train()

    return mixed, labels

def mixing_prob(epoch, cfg):
    """Flat probability after warmup — no cosine decay back to zero."""
    return 0.0 if epoch<=cfg['warmup_epochs'] else cfg['p_flat']

# Per-dataset configs — tuned from ablation results
DATASET_CFG = {
    'aircraft': dict(warmup_epochs=60, p_flat=0.15, keep_ratio=0.50,
                     s_min=0.20, s_max=0.45, filter_epoch=60,
                     min_conf=0.75, K=2, noise_scale=0.008),
    'cub':      dict(warmup_epochs=8,  p_flat=0.35, keep_ratio=0.32,
                     s_min=0.30, s_max=0.80, filter_epoch=8,
                     min_conf=0.60, K=3, noise_scale=0.015),
    'flowers':  dict(warmup_epochs=8,  p_flat=0.40, keep_ratio=0.28,
                     s_min=0.30, s_max=0.85, filter_epoch=8,
                     min_conf=0.55, K=3, noise_scale=0.018),
    'cars':     dict(warmup_epochs=50, p_flat=0.20, keep_ratio=0.45,
                     s_min=0.25, s_max=0.55, filter_epoch=50,
                     min_conf=0.70, K=2, noise_scale=0.010),
    'dogs':     dict(warmup_epochs=10, p_flat=0.30, keep_ratio=0.35,
                     s_min=0.30, s_max=0.75, filter_epoch=10,
                     min_conf=0.60, K=3, noise_scale=0.015),
    # Long-tail variants
    'cub_lt':   dict(warmup_epochs=10,  p_flat=0.30, keep_ratio=0.40,
                     s_min=0.20, s_max=0.65, filter_epoch=10,
                     min_conf=0.65, K=3, noise_scale=0.012),
    'flower_lt':dict(warmup_epochs=8,  p_flat=0.45, keep_ratio=0.28,
                     s_min=0.30, s_max=0.85, filter_epoch=8,
                     min_conf=0.50, K=3, noise_scale=0.018),
    # Few-shot variants (conservative — protect scarce samples)
    'cub_1shot': dict(warmup_epochs=5, p_flat=0.15, keep_ratio=0.55,
                      s_min=0.15, s_max=0.40, filter_epoch=5,
                      min_conf=0.80, K=2, noise_scale=0.006),
    'cub_5shot': dict(warmup_epochs=5, p_flat=0.25, keep_ratio=0.45,
                      s_min=0.20, s_max=0.55, filter_epoch=5,
                      min_conf=0.70, K=2, noise_scale=0.010),
    'cub_10shot':dict(warmup_epochs=5, p_flat=0.30, keep_ratio=0.40,
                      s_min=0.25, s_max=0.65, filter_epoch=5,
                      min_conf=0.65, K=3, noise_scale=0.012),
    'cub_all':   dict(warmup_epochs=8, p_flat=0.35, keep_ratio=0.35,
                      s_min=0.30, s_max=0.75, filter_epoch=8,
                      min_conf=0.60, K=3, noise_scale=0.015),
    # Waterbird (high fg_loss emphasis handled by high keep_ratio)
    'waterbird': dict(warmup_epochs=8, p_flat=0.35, keep_ratio=0.40,
                      s_min=0.25, s_max=0.70, filter_epoch=8,
                      min_conf=0.60, K=3, noise_scale=0.012),
}
print("HDMix core loaded ✓")

HDMix core loaded ✓


In [3]:
# ═══════════════════════════════════════════════════════════════════
# Dataset loaders — all 5 FGVC datasets + CUB few-shot + Waterbird
# ═══════════════════════════════════════════════════════════════════

PATHS = {
    'aircraft': '/kaggle/input/datasets/dedon002/this-data/fgvc-aircraft-2013b/fgvc-aircraft-2013b/data',
    'cub':      '/kaggle/input/datasets/dedon002/this-data/CUB_200_201.tgz',
    'flowers':  ('/kaggle/input/datasets/dedon002/this-data/102flowers.tgz',
                 '/kaggle/input/datasets/dedon002/this-data/imagelabels.mat'),
    'dogs':     '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset',
    'cars':     '/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder',
}

class ImgDataset(Dataset):
    def __init__(self,imgs,labels,tf):
        self.imgs=[Path(p) for p in imgs]; self.labels=labels; self.tf=tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self,i):
        return self.tf(Image.open(self.imgs[i]).convert('RGB')), self.labels[i]

def split_cls(imgs,lbls,frac=0.15,seed=42):
    rng=random.Random(seed); g={}
    for img,lbl in zip(imgs,lbls): g.setdefault(lbl,[]).append(img)
    ti,tl,vi,vl=[],[],[],[]
    for lbl,items in g.items():
        rng.shuffle(items); n=max(1,int(len(items)*frac))
        vi+=items[:n]; vl+=[lbl]*n
        ti+=items[n:]; tl+=[lbl]*(len(items)-n)
    return ti,tl,vi,vl

def make_loaders(ti,tl,vi,vl,ei,el,batch,train_tf,test_tf):
    kw=dict(num_workers=2,pin_memory=True)
    return (DataLoader(ImgDataset(ti,tl,train_tf),batch,shuffle=True,drop_last=True,**kw),
            DataLoader(ImgDataset(vi,vl,test_tf), batch,shuffle=False,**kw),
            DataLoader(ImgDataset(ei,el,test_tf), batch,shuffle=False,**kw))

def _extract_cub(dest='/tmp/cub_extract'):
    dest=Path(dest)
    if not any(dest.rglob('*.jpg')):
        dest.mkdir(parents=True,exist_ok=True)
        with tarfile.open(PATHS['cub'],'r:*') as t: t.extractall(dest)
    root=list(dest.rglob('images.txt'))[0].parent
    id2p,id2s,id2l={},{},{}
    for line in open(root/'images.txt'):
        i,p=line.strip().split(); id2p[i]=root/'images'/p
    for line in open(root/'train_test_split.txt'):
        i,s=line.strip().split(); id2s[i]=int(s)
    for line in open(root/'image_class_labels.txt'):
        i,c=line.strip().split(); id2l[i]=int(c)-1
    return id2p,id2s,id2l

def load_aircraft(use_trainval=True):
    """use_trainval=True trains on train+val (same as SGD-Mix paper)."""
    root=Path(PATHS['aircraft'])
    def rd(sp):
        imgs,lbls=[],[]
        with open(root/f'images_variant_{sp}.txt') as f:
            for line in f:
                p=line.strip().split(maxsplit=1)
                if len(p)==2:
                    imgs.append(root/'images'/f'{p[0]}.jpg')
                    lbls.append(p[1].strip())
        return imgs,lbls
    ti,tl=rd('train'); vi,vl=rd('val'); ei,el=rd('test')
    cls=sorted(set(tl+vl+el)); c2i={c:i for i,c in enumerate(cls)}
    ti_=[str(x) for x in ti]; tl_=[c2i[l] for l in tl]
    vi_=[str(x) for x in vi]; vl_=[c2i[l] for l in vl]
    ei_=[str(x) for x in ei]; el_=[c2i[l] for l in el]
    if use_trainval:
        # Combine train+val; hold out 1/class for early-stopping signal
        full_i=ti_+vi_; full_l=tl_+vl_
        rng=random.Random(42); c2idx={}
        for i,(img,lbl) in enumerate(zip(full_i,full_l)):
            c2idx.setdefault(lbl,[]).append(i)
        val_idx=set()
        for lbl,idxs in c2idx.items():
            rng.shuffle(idxs); val_idx.update(idxs[:1])
        tr_idx=[i for i in range(len(full_i)) if i not in val_idx]
        ti_=[full_i[i] for i in tr_idx]; tl_=[full_l[i] for i in tr_idx]
        vi_=[full_i[i] for i in sorted(val_idx)]
        vl_=[full_l[i] for i in sorted(val_idx)]
    return ti_,tl_,vi_,vl_,ei_,el_,len(cls)

def load_cub():
    id2p,id2s,id2l=_extract_cub()
    ids=sorted(id2p)
    tr_all=[i for i in ids if id2s[i]==1]; te=[i for i in ids if id2s[i]==0]
    ti,tl,vi,vl=split_cls([id2p[i] for i in tr_all],[id2l[i] for i in tr_all])
    return ([str(x) for x in ti],tl,[str(x) for x in vi],vl,
            [str(id2p[i]) for i in te],[id2l[i] for i in te],max(id2l.values())+1)

def load_flowers():
    dest=Path('/tmp/flowers_extract')
    if not any(dest.rglob('*.jpg')):
        dest.mkdir(parents=True,exist_ok=True)
        with tarfile.open(PATHS['flowers'][0],'r:*') as t: t.extractall(dest)
    jpgs=sorted(dest.rglob('*.jpg'))
    lbls=(scipy.io.loadmat(PATHS['flowers'][1])['labels'].flatten()-1).tolist()
    rng=random.Random(42); cls2f={}
    for f,l in zip(jpgs,lbls): cls2f.setdefault(int(l),[]).append(f)
    ti,tl,vi,vl,ei,el=[],[],[],[],[],[]
    for l,files in cls2f.items():
        rng.shuffle(files)
        ti+=files[:10]; tl+=[l]*10
        vi+=files[10:20]; vl+=[l]*10
        ei+=files[20:]; el+=[l]*len(files[20:])
    return ([str(x) for x in ti],tl,[str(x) for x in vi],vl,
            [str(x) for x in ei],el,102)

def load_dogs():
    root=Path(PATHS['dogs']); img_root=root/'images'/'Images'
    if not img_root.exists(): img_root=list(root.rglob('Images'))[0]
    cls_names=sorted([d.name for d in img_root.iterdir() if d.is_dir()])
    c2i={c:i for i,c in enumerate(cls_names)}
    all_imgs,all_lbls=[],[]
    for cd in sorted(img_root.iterdir()):
        if not cd.is_dir(): continue
        l=c2i[cd.name]
        for jpg in sorted(cd.glob('*.jpg')): all_imgs.append(str(jpg)); all_lbls.append(l)
    rng=random.Random(42); c2imgs={}
    for img,lbl in zip(all_imgs,all_lbls): c2imgs.setdefault(lbl,[]).append(img)
    ti,tl,vi,vl,ei,el=[],[],[],[],[],[]
    for lbl,items in c2imgs.items():
        rng.shuffle(items)
        nt=max(1,int(len(items)*0.20)); nv=max(1,int(len(items)*0.15))
        ei+=items[:nt]; el+=[lbl]*nt
        vi+=items[nt:nt+nv]; vl+=[lbl]*nv
        ti+=items[nt+nv:]; tl+=[lbl]*(len(items)-nt-nv)
    return ti,tl,vi,vl,ei,el,len(cls_names)

def load_cars():
    root=Path(PATHS['cars'])
    tr_dir=root/'car_data'/'car_data'/'train'
    te_dir=root/'car_data'/'car_data'/'test'
    cls=sorted([d.name for d in tr_dir.iterdir() if d.is_dir()]); c2i={c:i for i,c in enumerate(cls)}
    def scan(d):
        imgs,lbls=[],[]
        for cd in sorted(d.iterdir()):
            if not cd.is_dir(): continue
            l=c2i.get(cd.name,0)
            for jpg in cd.glob('*.jpg'): imgs.append(str(jpg)); lbls.append(l)
        return imgs,lbls
    ti_r,tl=scan(tr_dir); ei,el=scan(te_dir)
    ti,tl,vi,vl=split_cls(ti_r,tl)
    return ti,tl,vi,vl,ei,el,len(cls)

def load_cub_fewshot(shot):
    id2p,id2s,id2l=_extract_cub()
    ids=sorted(id2p)
    tr_all=[i for i in ids if id2s[i]==1]
    te=[i for i in ids if id2s[i]==0]
    rng=random.Random(42); c2tr={}
    for i in tr_all: c2tr.setdefault(id2l[i],[]).append(i)
    ti,tl=[],[]
    for cls_idx,iids in c2tr.items():
        rng.shuffle(iids)
        keep=iids if shot=='all' else iids[:shot]
        for iid in keep: ti.append(str(id2p[iid])); tl.append(cls_idx)
    ti,tl,vi,vl=split_cls(ti,tl,frac=0.15)
    ei=[str(id2p[i]) for i in te]; el=[id2l[i] for i in te]
    NC=max(id2l.values())+1
    return ti,tl,vi,vl,ei,el,NC

def load_waterbird():
    id2p,id2s,id2l=_extract_cub()
    dest=Path('/tmp/cub_extract')
    root=list(dest.rglob('images.txt'))[0].parent
    cls_names={}
    for line in open(root/'classes.txt'):
        c,n=line.strip().split(maxsplit=1); cls_names[int(c)-1]=n.lower()
    water_kw=['albatross','tern','gull','pelican','cormorant','gannet',
              'frigatebird','jaeger','petrel','booby','shearwater',
              'duck','goose','heron','egret','kingfisher','grebe',
              'loon','merganser','crane']
    wb_cls={i for i,n in cls_names.items() if any(k in n for k in water_kw)}
    def is_water_bg(p):
        n=Path(p).parent.name.lower()
        return any(k in n for k in ['water','ocean','lake','river','sea','beach','coast','pond'])
    ids=sorted(id2p)
    tr_all=[i for i in ids if id2s[i]==1]
    te=[i for i in ids if id2s[i]==0]
    ti_r=[str(id2p[i]) for i in tr_all]
    tl_r=[1 if id2l[i] in wb_cls else 0 for i in tr_all]
    ti,tl,vi,vl=split_cls(ti_r,tl_r)
    te_imgs=[str(id2p[i]) for i in te]
    te_lbls=[1 if id2l[i] in wb_cls else 0 for i in te]
    bird_g=[1 if id2l[i] in wb_cls else 0 for i in te]
    bg_g=[1 if is_water_bg(str(id2p[i])) else 0 for i in te]
    groups=list(zip(bird_g,bg_g))
    return ti,tl,vi,vl,te_imgs,te_lbls,groups,2

def make_longtail(imgs, lbls, imbalance_factor, seed=42):
    """
    CUB-LT/Flower-LT correct convention.
    Since CUB's natural per-class count (~30) never exceeds the universal
    Many>100/Medium 20-100/Few<20 thresholds from CIFAR-LT/ImageNet-LT,
    those absolute thresholds cannot work here — Many would always be empty.
    Instead, classes are ranked by their ORIGINAL (pre-imbalance) frequency
    and split into thirds: top third = Many (head), middle third = Medium,
    bottom third = Few (tail). This is the standard small-dataset long-tail
    convention and is what makes the paper's CUB-LT Many=79%+ numbers possible.
    """
    rng = random.Random(seed)
    c2imgs = {}
    for img, lbl in zip(imgs, lbls):
        c2imgs.setdefault(lbl, []).append(img)

    # Rank classes by ORIGINAL frequency (head -> tail)
    ranked = sorted(c2imgs.keys(), key=lambda k: -len(c2imgs[k]))
    n_cls  = len(ranked)
    max_n  = len(c2imgs[ranked[0]])
    min_n  = max(1, max_n // imbalance_factor)

    new_imgs, new_lbls = [], []
    for rank, cls_idx in enumerate(ranked):
        items = c2imgs[cls_idx]
        mu = max_n * (min_n / max_n) ** (rank / max(n_cls - 1, 1))
        n  = max(1, min(int(round(mu)), len(items)))
        rng.shuffle(items)
        new_imgs += items[:n]
        new_lbls += [cls_idx] * n

    # Bucket by rank-third, NOT by absolute post-imbalance count
    third      = n_cls // 3
    many_cls   = set(ranked[:third])
    medium_cls = set(ranked[third:2*third])
    few_cls    = set(ranked[2*third:])

    return new_imgs, new_lbls, many_cls, medium_cls, few_cls


print("Dataset loaders ✓")

Dataset loaders ✓


In [4]:
# ═══════════════════════════════════════════════════════════════════
# Transforms — one factory per image size
# ═══════════════════════════════════════════════════════════════════

def _norm(): return T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

def train_tf(sz):
    return T.Compose([
        T.RandomResizedCrop(sz,scale=(0.40,1.0)),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.05),
        T.RandAugment(num_ops=2,magnitude=10),
        T.ColorJitter(0.3,0.3,0.3,0.1),
        T.ToTensor(), _norm(),
        T.RandomErasing(p=0.20,scale=(0.02,0.12)),
    ])

def test_tf(sz):
    return T.Compose([T.Resize((sz,sz)), T.ToTensor(), _norm()])

def tta_tfs(sz):
    """5 crops × 3 scales = 15 views."""
    norm=_norm(); tfs=[]
    for scale in [1.0,1.1,1.15]:
        s=int(sz*scale)
        tfs+=[
            T.Compose([T.Resize((s,s)),T.CenterCrop(sz),T.ToTensor(),norm]),
            T.Compose([T.Resize((s,s)),T.CenterCrop(sz),
                       T.RandomHorizontalFlip(p=1.0),T.ToTensor(),norm]),
        ]
    tfs+=[
        T.Compose([T.Resize((sz,sz)),T.ToTensor(),norm]),
        T.Compose([T.Resize((sz,sz)),T.RandomHorizontalFlip(p=1.0),T.ToTensor(),norm]),
        T.Compose([T.Resize((int(sz*1.2),int(sz*1.2))),
                   T.FiveCrop(sz),T.Lambda(lambda c:c[0]),T.ToTensor(),norm]),
    ]
    return tfs

print("Transforms ✓")

Transforms ✓


In [ ]:

import torch.nn as nn

def make_model(backbone, nc, device):
    """
    'resnet50'           → timm resnet50.a1_in1k  (ResNet Strikes Back, +4pt vs default)
    'vit_base_patch16_384' → timm vit_base_patch16_384.orig_in21k_ft_in1k
    """
    if backbone == 'resnet50':
        m = timm.create_model('resnet50.a1_in1k', pretrained=True,
                               num_classes=nc, drop_rate=0.0)
    elif 'vit' in backbone:
        m = timm.create_model('vit_base_patch16_384.orig_in21k_ft_in1k',
                               pretrained=True, num_classes=nc, drop_rate=0.0)
    else:
        m = timm.create_model(backbone, pretrained=True,
                               num_classes=nc, drop_rate=0.0)
    return m.to(device)

def make_optimizer(model, lr_bb=1e-4, lr_head=1e-3, wd=1e-4):
    """10x LR ratio backbone:head — correct fine-tuning ratio."""
    bb,hd = split_params(model)
    return torch.optim.AdamW([
        {'params':bb, 'lr':lr_bb},
        {'params':hd, 'lr':lr_head},
    ], betas=(0.9,0.999), eps=1e-8, weight_decay=wd)

def make_scheduler(optimizer, epochs):
    def lr_lambda(ep):
        if ep<5: return (ep+1)/5
        t=(ep-5)/max(1,epochs-5)
        return max(1e-6, 0.5*(1+np.cos(np.pi*t)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def train_one_epoch(model, loader, optimizer, scaler, epoch,
                    total_epochs, cfg, device, class_weight=None, label_smooth=0.05):
    model.train()
    loss_sum,correct,total=0.,0,0
    p_mix=mixing_prob(epoch,cfg)
    for imgs,labels in loader:
        imgs=imgs.to(device,non_blocking=True)
        labels=labels.to(device,non_blocking=True)
        optimizer.zero_grad()
        if p_mix>0 and random.random()<p_mix:
            imgs,labels=hdmix(imgs,labels,model,epoch,total_epochs,cfg)
        with autocast():
            out=model(imgs)
            loss=nn.functional.cross_entropy(out,labels,
                                              weight=class_weight,
                                              label_smoothing=label_smooth)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
        scaler.step(optimizer); scaler.update()
        loss_sum+=loss.item()*imgs.size(0)
        correct+=(out.argmax(1)==labels).sum().item()
        total+=imgs.size(0)
    return loss_sum/total, correct/total, p_mix

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval(); correct,total=0,0
    for imgs,labels in loader:
        imgs=imgs.to(device,non_blocking=True)
        labels=labels.to(device,non_blocking=True)
        with autocast(): out=model(imgs)
        correct+=(out.argmax(1)==labels).sum().item(); total+=imgs.size(0)
    return correct/total

@torch.no_grad()
def evaluate_tta(model, ei, el, sz, batch, device, save_dir):
    ckpt=torch.load(f'{save_dir}/best.pth',map_location=device)
    model.load_state_dict(ckpt['model']); model.eval()
    all_probs=[]
    for tf in tta_tfs(sz):
        ds=ImgDataset(ei,el,tf)
        ld=DataLoader(ds,batch,shuffle=False,num_workers=2,pin_memory=True)
        probs=[]
        for imgs,_ in ld:
            with autocast(): probs.append(torch.softmax(model(imgs.to(device)),1).cpu())
        all_probs.append(torch.cat(probs))
    preds=torch.stack(all_probs).mean(0).argmax(1)
    return (preds==torch.tensor(el)).float().mean().item()

def run_training(model, train_loader, val_loader, optimizer, scheduler,
                 scaler, epochs, cfg, device, save_dir,
                 class_weight=None, label_smooth=0.05):
    best_val=0.
    for epoch in range(1,epochs+1):
        t0=time.time()
        tr_loss,tr_acc,p_mix=train_one_epoch(
            model,train_loader,optimizer,scaler,epoch,epochs,
            cfg,device,class_weight,label_smooth)
        va_acc=evaluate(model,val_loader,device)
        scheduler.step()
        if va_acc>best_val:
            best_val=va_acc
            torch.save({'model':model.state_dict()},f'{save_dir}/best.pth')
            marker=' ← best'
        else: marker=''
        phase='CLEAN' if p_mix==0 else f'MIX p={p_mix:.2f}'
        print(f"Ep {epoch:3d}/{epochs} [{phase:<12}] loss={tr_loss:.4f} "
              f"tr={tr_acc*100:.2f}% val={va_acc*100:.2f}% | "
              f"{time.time()-t0:.0f}s{marker}")
    return best_val

print("Training utilities ✓")

Training utilities ✓


## EXPERIMENT 1 — Fine-Grained Classification

Change `DATASET` to `'aircraft'` | `'cub'` | `'flowers'` | `'dogs'` | `'cars'` and run this section.

In [6]:
# # ── Config — change DATASET here
# DATASET    = 'cars'
# BACKBONE   = 'resnet50'
# EPOCHS     = 100
# BATCH      = 32
# IMAGE_SIZE = 448
# SAVE_DIR   = f'/kaggle/working/exp1_{DATASET}'
# os.makedirs(SAVE_DIR, exist_ok=True)

# CFG = DATASET_CFG[DATASET]
# print(f"Exp1: {DATASET.upper()} | backbone={BACKBONE} | img={IMAGE_SIZE} | epochs={EPOCHS}")
# print(f"CFG: {CFG}")

In [7]:
# # ── Load data
# fn={'aircraft':load_aircraft,'cub':load_cub,
#     'flowers':load_flowers,'dogs':load_dogs,'cars':load_cars}
# if DATASET=='aircraft':
#     ti,tl,vi,vl,ei,el,NC = load_aircraft(use_trainval=True)
# else:
#     ti,tl,vi,vl,ei,el,NC = fn[DATASET]()

# train_loader,val_loader,test_loader = make_loaders(
#     ti,tl,vi,vl,ei,el,BATCH,train_tf(IMAGE_SIZE),test_tf(IMAGE_SIZE))
# print(f"Train:{len(ti)} Val:{len(vi)} Test:{len(ei)} Classes:{NC}")

In [8]:
# # ── Model + optimizer + scheduler
# model     = make_model(BACKBONE, NC, DEVICE)
# optimizer = make_optimizer(model, lr_bb=1e-4, lr_head=1e-3)
# scheduler = make_scheduler(optimizer, EPOCHS)
# scaler    = GradScaler()
# print(f"{BACKBONE} | Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [9]:
# # ── Train
# best_val = run_training(
#     model, train_loader, val_loader,
#     optimizer, scheduler, scaler,
#     EPOCHS, CFG, DEVICE, SAVE_DIR,
#     label_smooth=0.05
# )

In [10]:
# # ── Evaluate with TTA
# PAPER_E1 = {'cub':87.66,'aircraft':90.16,'flowers':99.59,'cars':95.27,'dogs':88.01}
# acc   = evaluate_tta(model, ei, el, IMAGE_SIZE, BATCH, DEVICE, SAVE_DIR)
# delta = acc*100 - PAPER_E1[DATASET]
# print(f"\n{'='*55}")
# print(f"  Dataset   : {DATASET.upper()}")
# print(f"  HDMix TTA : {acc*100:.2f}%")
# print(f"  SGD-Mix   : {PAPER_E1[DATASET]:.2f}%")
# print(f"  Delta     : {delta:+.2f}%  {'✓ BEATS PAPER' if delta>0 else '✗ below paper'}")
# print(f"{'='*55}")
# key={'aircraft':'Aircraft','cub':'CUB','flowers':'Flowers','dogs':'Dogs','cars':'Cars'}[DATASET]
# RESULTS['exp1'][key] = round(acc*100,2)
# print(f"Saved → RESULTS['exp1']['{key}'] = {RESULTS['exp1'][key]}")

In [11]:
# # ── Plot Exp1 (run after all 5 datasets)
# paper_e1 = {'CUB':87.66,'Aircraft':90.16,'Flowers':99.59,'Cars':95.27,'Dogs':88.01}
# base_e1  = {'CUB':86.64,'Aircraft':89.09,'Flowers':99.27,'Cars':94.54,'Dogs':87.48}
# your_e1  = RESULTS['exp1']
# datasets = list(paper_e1.keys()); x=np.arange(len(datasets)); w=0.25
# fig,ax=plt.subplots(figsize=(13,6))
# b1=ax.bar(x-w, [base_e1[d]  for d in datasets],w,label='Baseline',  color='#aec6cf',edgecolor='k')
# b2=ax.bar(x,   [paper_e1[d] for d in datasets],w,label='SGD-Mix',   color='#4a90d9',edgecolor='k')
# b3=ax.bar(x+w, [your_e1[d]  for d in datasets],w,label='HDMix(Ours)',color='#e05c5c',edgecolor='k')
# for bars in [b1,b2,b3]:
#     for bar in bars:
#         h=bar.get_height()
#         if h>0: ax.annotate(f'{h:.1f}',xy=(bar.get_x()+bar.get_width()/2,h),
#                             xytext=(0,3),textcoords='offset points',
#                             ha='center',va='bottom',fontsize=8)
# ax.set_xticks(x); ax.set_xticklabels(datasets,fontsize=12)
# ax.set_ylim(80,102); ax.legend(fontsize=11)
# ax.set_ylabel('Top-1 Accuracy (%)'); ax.set_title('Exp1: Fine-Grained Classification',fontsize=13)
# ax.yaxis.grid(True,alpha=0.4); ax.set_axisbelow(True)
# plt.tight_layout(); plt.savefig(f'{PLOT_DIR}/exp1.png',dpi=150,bbox_inches='tight')
# plt.show(); print("Plot saved ✓")

## EXPERIMENT 2 — Long-Tail Classification

Change `DATASET` to `'cub'` | `'flowers'` and `IMBALANCE` to `100` | `50` | `10`.

In [12]:
# ── Config
DATASET   = 'flowers'    # 'cub' | 'flowers'
IMBALANCE = 10       # 100 | 50 | 10
BACKBONE  = 'resnet50'
EPOCHS    = 100; BATCH=32; IMAGE_SIZE=448
SAVE_DIR  = f'/kaggle/working/exp2_{DATASET}_IF{IMBALANCE}'
os.makedirs(SAVE_DIR, exist_ok=True)

CFG_LT = DATASET_CFG['cub_lt' if DATASET=='cub' else 'flower_lt']
print(f"Exp2: {DATASET.upper()} IF={IMBALANCE} | CFG: {CFG_LT}")

Exp2: FLOWERS IF=10 | CFG: {'warmup_epochs': 8, 'p_flat': 0.45, 'keep_ratio': 0.28, 's_min': 0.3, 's_max': 0.85, 'filter_epoch': 8, 'min_conf': 0.5, 'K': 3, 'noise_scale': 0.018}


In [13]:
if DATASET=='cub':     _ti,_tl,_vi,_vl,ei,el,NC = load_cub()
else:                  _ti,_tl,_vi,_vl,ei,el,NC = load_flowers()

ti_lt, tl_lt, many_cls, medium_cls, few_cls = make_longtail(_ti, _tl, IMBALANCE)

print(f"Original train:{len(_ti)} → LT train:{len(ti_lt)}")
print(f"Many:{len(many_cls)} Medium:{len(medium_cls)} Few:{len(few_cls)} classes")
print(f"LT count range: {min(Counter(tl_lt).values())} – {max(Counter(tl_lt).values())}")

cnt_lt = Counter(tl_lt)
freq   = torch.tensor([cnt_lt.get(i,1) for i in range(NC)], dtype=torch.float)
weight = (1.0/freq); weight=(weight/weight.sum()*NC).to(DEVICE)

kw=dict(num_workers=2,pin_memory=True)
train_loader=DataLoader(ImgDataset(ti_lt,tl_lt,train_tf(IMAGE_SIZE)),BATCH,shuffle=True,drop_last=True,**kw)
val_loader  =DataLoader(ImgDataset(_vi,_vl,test_tf(IMAGE_SIZE)),BATCH,shuffle=False,**kw)
test_loader =DataLoader(ImgDataset(ei,el,test_tf(IMAGE_SIZE)),BATCH,shuffle=False,**kw)

Original train:1020 → LT train:400
Many:34 Medium:34 Few:34 classes
LT count range: 1 – 10


In [14]:
model     = make_model(BACKBONE, NC, DEVICE)
optimizer = make_optimizer(model, lr_bb=1e-4, lr_head=1e-3)
scheduler = make_scheduler(optimizer, EPOCHS)
scaler    = GradScaler()

best_val = run_training(
    model, train_loader, val_loader,
    optimizer, scheduler, scaler,
    EPOCHS, CFG_LT, DEVICE, SAVE_DIR,
    class_weight=weight, label_smooth=0.05
)

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Ep   1/100 [CLEAN       ] loss=4.8077 tr=0.78% val=0.88% | 17s ← best
Ep   2/100 [CLEAN       ] loss=4.7726 tr=1.56% val=2.55% | 15s ← best
Ep   3/100 [CLEAN       ] loss=4.7269 tr=3.39% val=3.73% | 15s ← best
Ep   4/100 [CLEAN       ] loss=4.6689 tr=8.07% val=13.63% | 15s ← best
Ep   5/100 [CLEAN       ] loss=4.5875 tr=11.20% val=23.14% | 15s ← best
Ep   6/100 [CLEAN       ] loss=4.4633 tr=26.56% val=32.65% | 14s ← best
Ep   7/100 [CLEAN       ] loss=4.3115 tr=39.84% val=40.78% | 15s ← best
Ep   8/100 [CLEAN       ] loss=4.0974 tr=54.43% val=39.61% | 15s
Ep   9/100 [MIX p=0.45  ] loss=3.8486 tr=57.03% val=41.18% | 17s ← best
Ep  10/100 [MIX p=0.45  ] loss=3.4907 tr=70.57% val=43.33% | 15s ← best
Ep  11/100 [MIX p=0.45  ] loss=3.1484 tr=73.44% val=46.76% | 15s ← best
Ep  12/100 [MIX p=0.45  ] loss=2.8172 tr=71.35% val=47.94% | 16s ← best
Ep  13/100 [MIX p=0.45  ] loss=2.3597 tr=83.85% val=52.06% | 16s ← best
Ep  14/100 [MIX p=0.45  ] loss=2.0705 tr=84.64% val=54.31% | 16s ← best
Ep  15

In [15]:
# ── Evaluate splits
@torch.no_grad()
def evaluate_splits():
    ckpt=torch.load(f'{SAVE_DIR}/best.pth',map_location=DEVICE)
    model.load_state_dict(ckpt['model']); model.eval()
    all_preds,all_labels=[],[]
    for imgs,labels in test_loader:
        with autocast(): out=model(imgs.to(DEVICE))
        all_preds.append(out.argmax(1).cpu()); all_labels.append(labels)
    preds=torch.cat(all_preds); targets=torch.cat(all_labels)
    acc_all=(preds==targets).float().mean().item()*100
    def acc(mask):
        if mask.sum()==0: return 0.
        return (preds[mask]==targets[mask]).float().mean().item()*100
    many_m  =torch.tensor([l.item() in many_cls   for l in targets])
    med_m   =torch.tensor([l.item() in medium_cls for l in targets])
    few_m   =torch.tensor([l.item() in few_cls    for l in targets])
    PAPER_LT={
        'cub':  {100:{'Many':84.34,'Medium':66.80,'Few':37.00,'All':49.48},
                  50:{'All':58.68}, 10:{'All':64.98}},
        'flowers':{100:{'Many':99.68,'Medium':96.30,'Few':79.98,'All':88.66},
                   50:{'All':94.03}, 10:{'All':96.89}},
    }
    print(f"\n{'='*62}")
    print(f"  Long-tail {DATASET.upper()} IF={IMBALANCE}")
    print(f"{'='*62}")
    print(f"  {'Split':<10} {'HDMix':>8}  {'SGD-Mix':>8}  {'Delta':>8}")
    p=PAPER_LT[DATASET][IMBALANCE]; results_d={}
    for sname,mask,key in [('Many',many_m,'Many'),('Medium',med_m,'Medium'),
                             ('Few',few_m,'Few'),('All',None,'All')]:
        ours=acc_all if mask is None else acc(mask)
        pap=p.get(key); pstr=f'{pap:.2f}%' if pap else '—'
        dstr=f'{ours-pap:+.2f}%' if pap else '—'
        beat='✓' if (pap and ours>pap) else '✗'
        print(f"  {sname:<10} {ours:>7.2f}%  {pstr:>8}  {dstr:>8}  {beat}")
        results_d[sname]=round(ours,2)
    print(f"{'='*62}")
    sk='exp2_cub' if DATASET=='cub' else 'exp2_flower'
    RESULTS[sk][IMBALANCE]=results_d
    print(f"Saved → RESULTS['{sk}'][{IMBALANCE}]")

evaluate_splits()


  Long-tail FLOWERS IF=10
  Split         HDMix   SGD-Mix     Delta
  Many         88.64%         —         —  ✗
  Medium       71.15%         —         —  ✗
  Few          49.43%         —         —  ✗
  All          76.76%    96.89%   -20.13%  ✗
Saved → RESULTS['exp2_flower'][10]
